# EY Frog Challenge CPU Baselines

Run this notebook on Kaggle CPU after the feature notebook. It reproduces the starter benchmark, trains the classical tabular baselines with spatial cross-validation, and can optionally create a CPU-side final submission from saved TPU artifacts.

In [ ]:
from pathlib import Path

def find_requirements_path() -> Path:
    candidates = [Path.cwd() / 'requirements-kaggle.txt', Path('/kaggle/working') / 'requirements-kaggle.txt']
    if Path('/kaggle/input').exists():
        candidates.extend(Path('/kaggle/input').glob('*/requirements-kaggle.txt'))
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Could not locate requirements-kaggle.txt')

REQUIREMENTS_PATH = find_requirements_path()
print(REQUIREMENTS_PATH)
!pip install -q -r "{REQUIREMENTS_PATH}"

In [ ]:
from pathlib import Path
import json
import sys

def find_project_root() -> Path:
    candidates = [Path.cwd(), Path('/kaggle/working')]
    if Path('/kaggle/input').exists():
        candidates.extend(Path('/kaggle/input').glob('*'))
    for candidate in candidates:
        if (candidate / 'baseline_models.py').exists() and (candidate / 'frog_challenge').exists():
            return candidate
    raise FileNotFoundError('Could not find project root containing baseline_models.py')

def find_tpu_artifact_root() -> Path | None:
    if not Path('/kaggle/input').exists():
        return None
    for candidate in Path('/kaggle/input').glob('*'):
        if (candidate / 'tpu_summary.json').exists():
            return candidate
    return None

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

FEATURE_DIR = Path('/kaggle/working/artifacts/features')
BASELINE_DIR = Path('/kaggle/working/artifacts/baselines')
TPU_ARTIFACT_DIR = find_tpu_artifact_root()
PROJECT_ROOT, FEATURE_DIR, BASELINE_DIR, TPU_ARTIFACT_DIR

In [ ]:
!python "{PROJECT_ROOT / 'baseline_models.py'}" --feature-dir "{FEATURE_DIR}" --output-dir "{BASELINE_DIR}"

In [ ]:
baseline_summary = json.loads((BASELINE_DIR / 'baseline_summary.json').read_text())
baseline_summary['best_model_name'], baseline_summary['best_oof_f1']

In [ ]:
if TPU_ARTIFACT_DIR is not None and TPU_ARTIFACT_DIR.exists():
    !python "{PROJECT_ROOT / 'baseline_models.py'}" --feature-dir "{FEATURE_DIR}" --output-dir "{BASELINE_DIR}" --tpu-artifact-dir "{TPU_ARTIFACT_DIR}"
    final_selection = json.loads((BASELINE_DIR / 'final_selection.json').read_text())
    final_selection
else:
    print('Mount a dataset containing tpu_summary.json to /kaggle/input to run CPU-side final selection.')